# 20-Domain Retention Campaign

**BASE vs ANCHOR generation across 20 diverse domains**

Тестируем anchor bias на удержание семантических ограничений при свободной генерации.

Домены: dietary (vegan, halal, gluten-free), code (FastAPI, Rust, PostgreSQL, TypeScript, K8s),
math (proof), legal (GDPR), medical (drug-free pain), academic style, zero-waste, organic farming,
budget travel, minimalist design, functional programming, metric units, Python typed.

---
Runtime: **GPU (T4 / A100 / L4)**  
Время: ~30-60 мин на все 20 доменов  
Модель: Qwen3.5-4B (та же что в geometry probe)

## 0. GPU Check

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('GPU not found — Runtime -> Change runtime type -> GPU')

## 1. Clone repo

In [ ]:
import os, shutil

REPO_URL = 'https://github.com/kharkilirov1/Anchor-engine.git'
BRANCH = 'main'
REPO_DIR = '/content/ABPT'

# Always fresh clone to avoid stale cache
if os.path.exists(REPO_DIR):
    print('Removing old clone...')
    shutil.rmtree(REPO_DIR)

print(f'Cloning branch {BRANCH}...')
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
!git log --oneline -3

## 2. Install dependencies

In [ ]:
%%time
!pip install -q --upgrade pip
!pip install -q torch torchvision accelerate einops scipy numpy pytest
!pip install -q "transformers @ git+https://github.com/huggingface/transformers.git@main"
print('Dependencies installed')

## 3. Quick test

In [ ]:
!python -m pytest tests/test_qwen_generation_bias.py -v --tb=short

## 4. Preview: list all 20 domains

In [ ]:
import sys
sys.path.insert(0, '.')
from src.data.retention_domains import RETENTION_DOMAINS

print(f'Total domains: {len(RETENTION_DOMAINS)}')
print()
print(f'{"#":>2} {"Domain":<35} {"Profile":<14} {"+ kw":>5} {"- kw":>5} {"Tokens":>6}')
print('-' * 75)
for i, d in enumerate(RETENTION_DOMAINS, 1):
    print(f'{i:>2} {d.name:<35} {d.bias_profile_name:<14} {len(d.positive_keywords):>5} {len(d.negative_keywords):>5} {d.max_new_tokens:>6}')

## 5. Geometry-Gated Campaign (Attention-Based Anchor Detection)

Для каждого домена:
1. Forward pass с `output_attentions=True`
2. **Attention mass** от последнего токена определяет anchor span автоматически
3. r1 profile по слоям → классификация: **flat** → ANCHOR bias, **mature/template** → BASE
4. Fallback на текстовый match если attentions недоступны

In [ ]:
%%time
# Geometry-gated campaign — classify first, then decide
!python scripts/run_qwen_20domain_geometry_gated.py \
    --model Qwen/Qwen3.5-4B \
    --max_new_tokens 500 \
    --device cuda

## 6. Results

In [ ]:
import json
from pathlib import Path

gated_dir = Path('archive/20domain_geometry_gated')
gated_files = sorted(gated_dir.glob('gated_campaign_*.json'),
                      key=lambda p: p.stat().st_mtime, reverse=True)

if not gated_files:
    print('No gated results yet. Run the geometry-gated campaign first.')
else:
    data = json.loads(gated_files[0].read_text())
    gs = data['geometry_summary']
    gt = data['gating_summary']
    
    print(f"Model: {data['model']}")
    print(f"Time: {data['total_time_s']:.0f}s")
    print()
    print("GEOMETRY CLUSTERS:")
    print(f"  flat (anchor needed):  {gs['flat']}")
    print(f"  mature (model ok):     {gs['mature']}")
    print(f"  template (model ok):   {gs['template']}")
    print(f"  unknown:               {gs['unknown']}")
    print()
    print("GATING RESULTS:")
    print(f"  Anchor invoked: {gt['anchor_invoked']}/{data['n_domains']}")
    print(f"  Base kept:      {gt['base_kept']}/{data['n_domains']}")
    print(f"  Anchor wins:    {gt['anchor_wins']}")
    print(f"  Anchor losses:  {gt['anchor_losses']}")
    print(f"  TOTAL LOSSES:   {gt['total_losses']} (was 14 without gating)")
    print()
    
    print(f'{"#":>2} {"Domain":<35} {"Cluster":<10} {"Route":<7} {"Chosen":<7} {"Base Q":>7} {"Out Q":>7} {"Delta":>7} {"Result":<6}')
    print('-' * 100)
    for i, r in enumerate(data['results'], 1):
        if 'error' in r:
            print(f"{i:>2} {r['domain']:<35} ERROR")
            continue
        bq = r.get('base_analysis', {}).get('quality_score', 0)
        cq = r.get('chosen_analysis', {}).get('quality_score', 0)
        tag = 'WIN' if r.get('gated_wins') else ('SAME' if r['chosen'] == 'base' else 'LOSS')
        print(f"{i:>2} {r['domain']:<35} {r['cluster']:<10} {r['route']:<7} {r['chosen']:<7} {bq:>7.1f} {cq:>7.1f} {r['delta_quality']:>+7.1f} {tag:<6}")

## 7. Download results

In [ ]:
import shutil, os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
out_name = f'abpt_20domain_full_{timestamp}'
out_dir = f'/tmp/{out_name}'
os.makedirs(out_dir, exist_ok=True)

# Copy both campaign results
for src in ['archive/20domain_campaign', 'archive/20domain_geometry_gated']:
    if os.path.exists(src):
        dest = f'{out_dir}/{os.path.basename(src)}'
        shutil.copytree(src, dest, dirs_exist_ok=True)

for f in ['playbook.md', 'research_state.json']:
    if os.path.exists(f):
        shutil.copy(f, out_dir)

shutil.make_archive(f'/tmp/{out_name}', 'zip', '/tmp', out_name)

try:
    from google.colab import files
    files.download(f'/tmp/{out_name}.zip')
    print(f'Downloaded: {out_name}.zip')
except ImportError:
    print(f'Saved: /tmp/{out_name}.zip')